![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Arabic–English LLM Evaluation
### Capability gaps, paired consistency, judge bias, and statistical rigor

A model can look strong on an English benchmark and still behave differently when the **same task** is asked in Arabic. In this lab, you will build a small, reproducible evaluation harness that treats language as an experimental variable rather than a cosmetic translation.


## What this lab covers

By the end of the notebook, you will be able to:

1. build a **paired bilingual evaluation set** where Arabic and English items test the same underlying skill;
2. run a local open-weight LLM with deterministic decoding and preserve its raw outputs;
3. measure **accuracy**, **Arabic–English performance gap**, and **pair consistency**;
4. use an exact paired test and a **paired bootstrap confidence interval** instead of comparing percentages alone;
5. probe an **LLM-as-a-judge** for position bias and judge-language sensitivity;
6. produce a compact evaluation card with enough metadata to reproduce the experiment.

> **Important:** the examples below form a *teaching micro-eval*, not a leaderboard benchmark. Research-scale Arabic evaluation should use curated resources such as ArabicMMLU, HELM Arabic, and DialectalArabicMMLU.


## Why paired evaluation?

If we ask unrelated questions in Arabic and English, a score difference may simply reflect different question difficulty. A **paired design** asks semantically matched versions of the same item.

For item \(i\), define:

\[
d_i = \mathbb{1}(\text{Arabic correct}) - \mathbb{1}(\text{English correct}).
\]

The mean of \(d_i\) is the Arabic-minus-English accuracy gap. Because each Arabic item is tied to an English counterpart, we can also inspect *which exact pairs disagree* and use paired statistical tests.


# Part 0 — Setup

The default model is **Qwen2.5-1.5B-Instruct**: small enough for a free Colab GPU while being substantially more useful for instruction following than the 0.5B variant. The evaluation harness is model-agnostic; you can replace it with another causal chat model later.


In [ ]:
!pip install -q -U "transformers>=4.45,<5" accelerate pandas numpy matplotlib scipy


In [ ]:
import json
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import binomtest
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_colwidth", 120)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("device candidates:", {
    "cuda": torch.cuda.is_available(),
    "mps": torch.backends.mps.is_available() if hasattr(torch.backends, "mps") else False,
})


# Part 1 — Build a paired bilingual micro-eval

Each row contains the same multiple-choice task in English and Modern Standard Arabic. We deliberately keep the **answer label identical** across languages, so the scorer does not need a second model.

The items span arithmetic, science, logic, and instruction following. This is enough to teach the method while keeping the notebook fast.


In [ ]:
PAIRED_ITEMS = [
    {"id":"math_01","category":"math","gold":"C",
     "prompt_en":"What is 17 × 6?\nA) 92\nB) 96\nC) 102\nD) 112\nReturn only A, B, C, or D.",
     "prompt_ar":"ما ناتج 17 × 6؟\nA) 92\nB) 96\nC) 102\nD) 112\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"math_02","category":"math","gold":"B",
     "prompt_en":"A car travels 180 km in 3 hours at a constant speed. What is its speed?\nA) 45 km/h\nB) 60 km/h\nC) 90 km/h\nD) 540 km/h\nReturn only A, B, C, or D.",
     "prompt_ar":"تقطع سيارة 180 كم خلال 3 ساعات بسرعة ثابتة. ما سرعتها؟\nA) 45 كم/س\nB) 60 كم/س\nC) 90 كم/س\nD) 540 كم/س\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"math_03","category":"math","gold":"D",
     "prompt_en":"What is three quarters of 28?\nA) 7\nB) 14\nC) 18\nD) 21\nReturn only A, B, C, or D.",
     "prompt_ar":"ما قيمة ثلاثة أرباع العدد 28؟\nA) 7\nB) 14\nC) 18\nD) 21\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"math_04","category":"math","gold":"D",
     "prompt_en":"What comes next in the sequence 2, 6, 12, 20, ?\nA) 22\nB) 24\nC) 28\nD) 30\nReturn only A, B, C, or D.",
     "prompt_ar":"ما العدد التالي في المتتالية 2، 6، 12، 20، ؟\nA) 22\nB) 24\nC) 28\nD) 30\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"science_01","category":"science","gold":"A",
     "prompt_en":"Which gas do plants primarily absorb during photosynthesis?\nA) Carbon dioxide\nB) Oxygen\nC) Nitrogen\nD) Helium\nReturn only A, B, C, or D.",
     "prompt_ar":"أي غاز تمتصه النباتات أساسًا أثناء عملية البناء الضوئي؟\nA) ثاني أكسيد الكربون\nB) الأكسجين\nC) النيتروجين\nD) الهيليوم\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"science_02","category":"science","gold":"C",
     "prompt_en":"At standard atmospheric pressure, pure water boils at approximately:\nA) 0°C\nB) 50°C\nC) 100°C\nD) 150°C\nReturn only A, B, C, or D.",
     "prompt_ar":"عند الضغط الجوي القياسي، يغلي الماء النقي تقريبًا عند:\nA) 0°م\nB) 50°م\nC) 100°م\nD) 150°م\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"science_03","category":"science","gold":"B",
     "prompt_en":"In DNA, adenine normally pairs with which base?\nA) Cytosine\nB) Thymine\nC) Guanine\nD) Uracil\nReturn only A, B, C, or D.",
     "prompt_ar":"في الحمض النووي DNA، ترتبط قاعدة الأدينين عادةً بأي قاعدة؟\nA) السيتوسين\nB) الثايمين\nC) الجوانين\nD) اليوراسيل\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"science_04","category":"science","gold":"A",
     "prompt_en":"What is the SI unit of force?\nA) Newton\nB) Joule\nC) Watt\nD) Pascal\nReturn only A, B, C, or D.",
     "prompt_ar":"ما وحدة القوة في النظام الدولي للوحدات؟\nA) نيوتن\nB) جول\nC) واط\nD) باسكال\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"logic_01","category":"logic","gold":"D",
     "prompt_en":"All tulips are flowers. No flowers are machines. Which statement must be true?\nA) Some machines are tulips\nB) All machines are flowers\nC) Some tulips are machines\nD) No tulips are machines\nReturn only A, B, C, or D.",
     "prompt_ar":"كل زهور التوليب أزهار، ولا توجد زهرة هي آلة. أي عبارة يجب أن تكون صحيحة؟\nA) بعض الآلات توليب\nB) كل الآلات أزهار\nC) بعض التوليب آلات\nD) لا توجد زهرة توليب هي آلة\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"logic_02","category":"logic","gold":"B",
     "prompt_en":"Maha is older than Lina. Lina is older than Noor. Who is the youngest?\nA) Maha\nB) Noor\nC) Lina\nD) Cannot be determined\nReturn only A, B, C, or D.",
     "prompt_ar":"مها أكبر سنًا من لينا، ولينا أكبر سنًا من نور. من الأصغر سنًا؟\nA) مها\nB) نور\nC) لينا\nD) لا يمكن تحديد ذلك\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"logic_03","category":"logic","gold":"C",
     "prompt_en":"If every researcher in a team knows Python, and Sara is a researcher in that team, what follows?\nA) Sara may not know Python\nB) Only Sara knows Python\nC) Sara knows Python\nD) Python is a researcher\nReturn only A, B, C, or D.",
     "prompt_ar":"إذا كان كل باحث في فريق ما يعرف لغة بايثون، وكانت سارة باحثة في ذلك الفريق، فما النتيجة؟\nA) قد لا تعرف سارة بايثون\nB) سارة وحدها تعرف بايثون\nC) سارة تعرف بايثون\nD) بايثون باحث\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"logic_04","category":"logic","gold":"A",
     "prompt_en":"A report must be submitted after review but before publication. Which order is valid?\nA) Review, then submit, then publish\nB) Submit, then review, then publish\nC) Publish, then review, then submit\nD) Submit, then publish, then review\nReturn only A, B, C, or D.",
     "prompt_ar":"يجب تسليم التقرير بعد المراجعة وقبل النشر. أي ترتيب صحيح؟\nA) مراجعة، ثم تسليم، ثم نشر\nB) تسليم، ثم مراجعة، ثم نشر\nC) نشر، ثم مراجعة، ثم تسليم\nD) تسليم، ثم نشر، ثم مراجعة\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"instr_01","category":"instruction","gold":"B",
     "prompt_en":"Which response follows the instruction 'reply with exactly three uppercase English letters'?\nA) ABCD\nB) KSA\nC) ksa\nD) K S A\nReturn only A, B, C, or D.",
     "prompt_ar":"أي استجابة تتبع التعليمات «أجب بثلاثة أحرف إنجليزية كبيرة بالضبط»؟\nA) ABCD\nB) KSA\nC) ksa\nD) K S A\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"instr_02","category":"instruction","gold":"C",
     "prompt_en":"Which output obeys 'give one integer and no other text'?\nA) The answer is 7\nB) 7.0\nC) 7\nD) seven\nReturn only A, B, C, or D.",
     "prompt_ar":"أي مخرج يلتزم بالتعليمات «أعط عددًا صحيحًا واحدًا دون أي نص آخر»؟\nA) The answer is 7\nB) 7.0\nC) 7\nD) seven\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"instr_03","category":"instruction","gold":"A",
     "prompt_en":"The instruction is 'return a JSON boolean, not a string'. Which output is valid?\nA) true\nB) \"true\"\nC) True\nD) yes\nReturn only A, B, C, or D.",
     "prompt_ar":"التعليمات هي «أعد قيمة منطقية بصيغة JSON وليست نصًا». أي مخرج صحيح؟\nA) true\nB) \"true\"\nC) True\nD) yes\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
    {"id":"instr_04","category":"instruction","gold":"D",
     "prompt_en":"Which response obeys 'answer with one word only'?\nA) Riyadh city\nB) It is Riyadh\nC) Riyadh Saudi\nD) Riyadh\nReturn only A, B, C, or D.",
     "prompt_ar":"أي استجابة تلتزم بالتعليمات «أجب بكلمة واحدة فقط»؟\nA) مدينة الرياض\nB) إنها الرياض\nC) الرياض السعودية\nD) الرياض\nأعد حرف الإجابة فقط: A أو B أو C أو D."},
]

eval_df = pd.DataFrame(PAIRED_ITEMS)
assert eval_df["id"].is_unique
assert set(eval_df["gold"]) <= set("ABCD")
assert eval_df[["prompt_en", "prompt_ar"]].notna().all().all()
print(f"Paired items: {len(eval_df)}")
display(eval_df[["id", "category", "gold"]])


## Experimental hygiene before inference

A bilingual comparison is only interpretable if the two sides are run under the same conditions. We therefore keep constant:

- model and model weights;
- chat template;
- decoding method (`do_sample=False`);
- maximum output length;
- scorer and answer labels;
- item order for the paired analysis.

We also retain **raw model text**. Never keep only the final score: parsing mistakes and formatting failures are themselves useful evaluation evidence.


# Part 2 — Load a local multilingual model

This lab does **not** require a paid API key. The model downloads once from Hugging Face. On a GPU it will run faster; CPU and Apple Silicon are also supported for this small model.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# CPU-only smoke-test alternative: "Qwen/Qwen2.5-0.5B-Instruct"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Loading", MODEL_ID, "on", DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto")
model.to(DEVICE)
model.eval()
# Qwen ships sampling defaults; clear them because this lab uses deterministic decoding.
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
print("Ready.")


In [ ]:
def generate_text(user_prompt, max_new_tokens=12):
    """Generate deterministically and return only newly generated text."""
    messages = [
        {"role": "system", "content": "Follow the user's requested output format exactly."},
        {"role": "user", "content": user_prompt},
    ]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def extract_choice(text):
    """Extract a standalone A/B/C/D choice; keep unparsable outputs as None."""
    match = re.search(r"(?<![A-Za-z])([ABCD])(?![A-Za-z])", text.upper())
    return match.group(1) if match else None

print(generate_text("What is 2 + 2? A) 3 B) 4 C) 5 D) 6. Return only A, B, C, or D."))


# Part 3 — Run the paired capability evaluation

`FAST_MODE=True` evaluates eight pairs, with two items from each category. Set it to `False` for all sixteen teaching items.

For research work, sixteen items are far too few. The point here is the **evaluation design**, not the absolute score.


In [ ]:
FAST_MODE = True
run_df = eval_df.groupby("category", group_keys=False).head(2).copy() if FAST_MODE else eval_df.copy()

records = []
for row in run_df.itertuples(index=False):
    for language in ("en", "ar"):
        raw = generate_text(getattr(row, f"prompt_{language}"))
        pred = extract_choice(raw)
        records.append({
            "id": row.id,
            "category": row.category,
            "language": language,
            "gold": row.gold,
            "prediction": pred,
            "correct": pred == row.gold,
            "raw_output": raw,
        })

results = pd.DataFrame(records)
display(results)


## Metric 1 — Accuracy is not enough

We report accuracy by language, but also compute **pair consistency**: how often the model selects the same option for matched Arabic and English items.

A model can have identical aggregate accuracy in both languages while failing on different examples. Pair consistency exposes that hidden instability.


In [ ]:
wide = results.pivot(index="id", columns="language", values=["prediction", "correct", "gold"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

acc = results.groupby("language")["correct"].mean()
acc_en = float(acc.get("en", np.nan))
acc_ar = float(acc.get("ar", np.nan))
pair_consistency = (wide["prediction_en"] == wide["prediction_ar"]).mean()
accuracy_gap = acc_ar - acc_en

summary = pd.DataFrame({
    "metric": ["English accuracy", "Arabic accuracy", "Arabic - English gap", "Pair consistency"],
    "value": [acc_en, acc_ar, accuracy_gap, pair_consistency],
})
display(summary.style.format({"value": "{:.1%}"}))


In [ ]:
both_correct = (wide["correct_en"] & wide["correct_ar"]).sum()
en_only = (wide["correct_en"] & ~wide["correct_ar"]).sum()
ar_only = (~wide["correct_en"] & wide["correct_ar"]).sum()
both_wrong = (~wide["correct_en"] & ~wide["correct_ar"]).sum()

paired_outcomes = pd.Series({
    "both correct": both_correct,
    "English only": en_only,
    "Arabic only": ar_only,
    "both wrong": both_wrong,
})
display(paired_outcomes.to_frame("items"))

ax = paired_outcomes.plot(kind="bar", title="Paired outcomes")
ax.set_ylabel("Number of item pairs")
plt.xticks(rotation=25, ha="right")
plt.show()


## Metric 2 — Exact paired test (McNemar logic)

For paired binary outcomes, only the **discordant pairs** carry evidence about a language difference:

- `English only` = English correct, Arabic wrong;
- `Arabic only` = Arabic correct, English wrong.

Under the null hypothesis that neither language has an advantage, either direction is equally likely. An exact two-sided binomial test on those discordant pairs gives the exact McNemar p-value.

With a tiny teaching set, expect low statistical power. A non-significant p-value does **not** prove equivalence.


In [ ]:
discordant = int(en_only + ar_only)
if discordant == 0:
    mcnemar_p = 1.0
else:
    mcnemar_p = binomtest(min(en_only, ar_only), n=discordant, p=0.5, alternative="two-sided").pvalue

print(f"English-only correct pairs: {en_only}")
print(f"Arabic-only correct pairs:  {ar_only}")
print(f"Exact paired p-value:       {mcnemar_p:.4f}")


## Metric 3 — Paired bootstrap confidence interval

A point estimate like “Arabic is 6 percentage points lower” hides uncertainty. We resample **item pairs**, not individual language rows, because Arabic and English observations from the same item are dependent.


In [ ]:
def paired_bootstrap_gap(wide_df, n_boot=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = wide_df["correct_ar"].astype(float).to_numpy() - wide_df["correct_en"].astype(float).to_numpy()
    n = len(diffs)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        sample_idx = rng.integers(0, n, size=n)
        boot[b] = diffs[sample_idx].mean()
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return diffs.mean(), lo, hi, boot

gap, ci_lo, ci_hi, boot = paired_bootstrap_gap(wide)
print(f"Arabic - English accuracy gap: {gap:+.1%}")
print(f"95% paired bootstrap CI:       [{ci_lo:+.1%}, {ci_hi:+.1%}]")

plt.hist(boot, bins=21)
plt.axvline(gap, linewidth=2)
plt.title("Paired bootstrap distribution of the language gap")
plt.xlabel("Arabic accuracy - English accuracy")
plt.ylabel("Bootstrap samples")
plt.show()


# Part 4 — Error analysis

Aggregate metrics tell us **whether** behavior differs. Error analysis asks **where and how**.

Start with language disagreements. Look at the raw output before blaming capability: sometimes the model knew the answer but violated the requested format, which is an instruction-following failure rather than a knowledge failure.


In [ ]:
mismatched_ids = wide.loc[wide["prediction_en"] != wide["prediction_ar"], "id"].tolist()
error_view = results[results["id"].isin(mismatched_ids)].sort_values(["id", "language"])

if len(error_view):
    display(error_view[["id", "category", "language", "gold", "prediction", "correct", "raw_output"]])
else:
    print("No Arabic–English prediction disagreements in this run.")


In [ ]:
category_accuracy = (
    results.groupby(["category", "language"])["correct"]
    .mean()
    .unstack("language")
    .rename(columns={"en": "English", "ar": "Arabic"})
)
display(category_accuracy.style.format("{:.0%}"))

category_accuracy.plot(kind="bar", title="Accuracy by category and language")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.show()


# Part 5 — LLM-as-a-judge: test the judge before trusting it

LLM judges are useful for open-ended evaluation, but the judge is also a model and can be biased. A classic failure mode is **position bias**: preference changes when candidate A and candidate B are swapped.

We create a tiny set where one answer is clearly better according to a simple rubric. We judge each pair twice: good answer first and good answer second. We also run the judge instruction in English and Arabic.

For teaching convenience, the **same small model loaded above acts as the judge**. For research or production evaluation, use an independently validated judge (or human labels), and report judge identity as part of the protocol.


In [ ]:
JUDGE_ITEMS = [
    {"id":"judge_01",
     "question_en":"Why do Earth seasons occur?",
     "question_ar":"لماذا تحدث فصول السنة على الأرض؟",
     "good_en":"Seasons are mainly caused by Earth's axial tilt as it orbits the Sun, changing sunlight angle and day length in each hemisphere.",
     "bad_en":"Seasons happen mainly because Earth is much closer to the Sun in summer and farther away in winter.",
     "good_ar":"تحدث الفصول أساسًا بسبب ميل محور الأرض أثناء دورانها حول الشمس، مما يغيّر زاوية أشعة الشمس وطول النهار في كل نصف من الكرة الأرضية.",
     "bad_ar":"تحدث الفصول أساسًا لأن الأرض تكون أقرب بكثير إلى الشمس في الصيف وأبعد عنها في الشتاء."},
    {"id":"judge_02",
     "question_en":"What does a confidence interval communicate?",
     "question_ar":"ماذا تعبّر فترة الثقة في التحليل الإحصائي؟",
     "good_en":"It summarizes uncertainty around an estimated quantity under a stated statistical procedure; it is not the probability that this particular fixed interval contains the parameter.",
     "bad_en":"It guarantees that the true value is inside the interval and removes uncertainty from the estimate.",
     "good_ar":"تلخص فترة الثقة عدم اليقين حول كمية مقدّرة وفق إجراء إحصائي محدد، ولا تعني أن هناك احتمالًا مباشرًا بأن المعلمة الثابتة تقع داخل هذه الفترة بعينها.",
     "bad_ar":"تضمن فترة الثقة أن القيمة الحقيقية موجودة داخل الفترة وتزيل عدم اليقين من التقدير."},
    {"id":"judge_03",
     "question_en":"Why should an evaluation preserve raw model outputs?",
     "question_ar":"لماذا ينبغي للاختبار الاحتفاظ بالمخرجات الخام للنموذج؟",
     "good_en":"Raw outputs make scoring auditable: they let us distinguish model errors from parser errors, inspect formatting failures, and reproduce later analyses.",
     "bad_en":"Raw outputs are unnecessary once a final accuracy number has been calculated.",
     "good_ar":"تجعل المخرجات الخام عملية التقييم قابلة للتدقيق؛ فهي تساعد على التمييز بين أخطاء النموذج وأخطاء المحلل، وفحص إخفاقات التنسيق، وإعادة التحليل لاحقًا.",
     "bad_ar":"لا حاجة للاحتفاظ بالمخرجات الخام بعد حساب رقم الدقة النهائي."},
    {"id":"judge_04",
     "question_en":"What is the fairest way to compare Arabic and English performance on the same capability?",
     "question_ar":"ما الطريقة الأكثر إنصافًا لمقارنة أداء العربية والإنجليزية في القدرة نفسها؟",
     "good_en":"Use semantically matched item pairs, keep inference settings fixed, preserve outputs, and analyze paired differences rather than unrelated language averages.",
     "bad_en":"Use any convenient Arabic questions and any convenient English questions, then subtract the two average scores.",
     "good_ar":"استخدم أزواجًا متطابقة دلاليًا من الأسئلة، وثبّت إعدادات الاستدلال، واحتفظ بالمخرجات، وحلّل الفروق الزوجية بدل مقارنة متوسطات مجموعتين مختلفتين من الأسئلة.",
     "bad_ar":"استخدم أي أسئلة عربية متاحة وأي أسئلة إنجليزية متاحة ثم اطرح متوسطي الدرجات."},
]
print("Judge items:", len(JUDGE_ITEMS))


In [ ]:
def make_judge_prompt(item, language="en", good_first=True):
    q = item[f"question_{language}"]
    good = item[f"good_{language}"]
    bad = item[f"bad_{language}"]
    a, b = (good, bad) if good_first else (bad, good)
    gold = "A" if good_first else "B"

    if language == "en":
        prompt = f"""You are evaluating two answers to the same question.
Choose the answer that is more correct, relevant, and directly responsive.
Do not prefer an answer merely because it is longer.

Question: {q}

Answer A: {a}
Answer B: {b}

Return only A or B."""
    else:
        prompt = f"""أنت تقيّم إجابتين عن السؤال نفسه.
اختر الإجابة الأكثر صحة وارتباطًا بالسؤال واستجابةً له مباشرة.
لا تفضّل إجابة لمجرد أنها أطول.

السؤال: {q}

الإجابة A: {a}
الإجابة B: {b}

أعد الحرف A أو B فقط."""
    return prompt, gold

judge_records = []
for item in JUDGE_ITEMS:
    for language in ("en", "ar"):
        for good_first in (True, False):
            prompt, gold = make_judge_prompt(item, language, good_first)
            raw = generate_text(prompt, max_new_tokens=8)
            pred = extract_choice(raw)
            judge_records.append({
                "id": item["id"],
                "judge_language": language,
                "good_first": good_first,
                "gold": gold,
                "prediction": pred,
                "correct": pred == gold,
                "raw_output": raw,
            })

judge_df = pd.DataFrame(judge_records)
display(judge_df)


## Measure judge accuracy and position sensitivity

For each item and judge language, compare the judge's *semantic preference* when the answer order is reversed.

If the judge selects A before the swap and still selects A after the swap, it has switched from the good answer to the bad answer. That is a position-sensitive flip, not stable evaluation.


In [ ]:
judge_accuracy = judge_df.groupby("judge_language")["correct"].mean()
display(judge_accuracy.rename({"en":"English judge prompt", "ar":"Arabic judge prompt"}).to_frame("accuracy").style.format("{:.1%}"))

jwide = judge_df.pivot(index=["id", "judge_language"], columns="good_first", values="prediction").reset_index()
jwide = jwide.rename(columns={True:"prediction_good_first", False:"prediction_good_second"})
jwide["choose_good_when_first"] = jwide["prediction_good_first"] == "A"
jwide["choose_good_when_second"] = jwide["prediction_good_second"] == "B"
jwide["position_sensitive"] = jwide["choose_good_when_first"] != jwide["choose_good_when_second"]

position_bias = jwide.groupby("judge_language")["position_sensitive"].mean()
display(position_bias.rename({"en":"English judge prompt", "ar":"Arabic judge prompt"}).to_frame("position-sensitive rate").style.format("{:.1%}"))


## Judge-language sensitivity

Now compare the *same answer pair and same ordering* under English versus Arabic judge instructions. If the selected candidate changes, the evaluation protocol itself is language-sensitive.

This does not automatically mean the judge is “biased” in a social sense. It means the **measurement instrument is unstable across prompt language**, which must be reported.


In [ ]:
lang_wide = judge_df.pivot(index=["id", "good_first"], columns="judge_language", values="prediction").reset_index()
lang_wide["judge_language_agreement"] = lang_wide["en"] == lang_wide["ar"]

print(f"English/Arabic judge-choice agreement: {lang_wide['judge_language_agreement'].mean():.1%}")
display(lang_wide)


# Part 6 — Produce a reproducible evaluation card

A useful evaluation result is more than a score. Record the model, decoding choices, seed, sample size, metrics, uncertainty, software versions, and raw-output retention policy.


## Export the raw evidence

Scores are derived artifacts. Save the underlying generations too, so another reviewer can audit parsing, formatting failures, and judge decisions. These files are runtime outputs; they are not part of the repository.


In [ ]:
results.to_json("arabic_english_eval_results.jsonl", orient="records", lines=True, force_ascii=False)
judge_df.to_json("arabic_english_judge_results.jsonl", orient="records", lines=True, force_ascii=False)
print("Saved raw capability and judge outputs as JSONL.")


In [ ]:
evaluation_card = {
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "device": DEVICE,
    "seed": SEED,
    "software": {
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "decoding": {"do_sample": False, "max_new_tokens_capability": 12, "max_new_tokens_judge": 8},
    "paired_items_run": int(wide.shape[0]),
    "metrics": {
        "accuracy_en": acc_en,
        "accuracy_ar": acc_ar,
        "accuracy_gap_ar_minus_en": accuracy_gap,
        "pair_consistency": float(pair_consistency),
        "mcnemar_exact_p": float(mcnemar_p),
        "paired_bootstrap_95ci": [float(ci_lo), float(ci_hi)],
    },
    "judge": {
        "items": len(JUDGE_ITEMS),
        "accuracy_by_prompt_language": {k: float(v) for k, v in judge_accuracy.items()},
        "position_sensitive_rate": {k: float(v) for k, v in position_bias.items()},
        "cross_language_choice_agreement": float(lang_wide["judge_language_agreement"].mean()),
    },
    "raw_outputs_retained": True,
    "note": "Teaching micro-eval; not a leaderboard benchmark.",
}

print(json.dumps(evaluation_card, ensure_ascii=False, indent=2))


# Part 7 — Student extensions

Choose one extension and justify the experimental design before running it:

1. **Scale the paired set.** Create at least 50 semantically matched Arabic–English items and report confidence intervals.
2. **Add a dialect carefully.** Add a Saudi Arabic condition, but separate *translation/adaptation quality* from model performance. Native-speaker review is strongly recommended.
3. **Change the model.** Compare an Arabic-centric model with a general multilingual model using exactly the same harness.
4. **Change the judge.** Use a different model as judge and measure whether conclusions survive the judge swap.
5. **Add perturbations.** Test punctuation, spelling, diacritics, or paraphrase robustness while keeping meaning fixed.
6. **Audit the parser.** Count outputs that were semantically correct but failed the strict answer format, and report them separately.

A strong write-up should distinguish **capability failure**, **format failure**, **translation/adaptation failure**, and **measurement failure**.


# Interpretation checklist

Before claiming that one language “performs worse,” ask:

- Are Arabic and English items genuinely matched in meaning and difficulty?
- Is the sample large enough for the uncertainty around the gap to be informative?
- Are disagreements concentrated in one category?
- Did the parser mis-handle any valid outputs?
- Does the conclusion survive a model change, prompt change, or judge-order swap?
- Are you measuring Modern Standard Arabic, a dialect, or a mixture?
- Have you preserved prompts, raw completions, model identifier, decoding settings, and code?

The goal of evaluation is not to manufacture a ranking. It is to make a **claim whose evidence can be inspected and reproduced**.


# References and further reading

- Liang et al. (2022/2023), **Holistic Evaluation of Language Models (HELM)** — https://arxiv.org/abs/2211.09110
- Zheng et al. (2023), **Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena** — https://arxiv.org/abs/2306.05685
- Koto et al. (2024), **ArabicMMLU: Assessing Massive Multitask Language Understanding in Arabic** — https://aclanthology.org/2024.findings-acl.334/
- Stanford CRFM (2025), **HELM Arabic** — https://crfm.stanford.edu/2025/12/18/helm-arabic.html
- Altakrori et al. (2026), **DialectalArabicMMLU: Benchmarking Dialectal Capabilities in Arabic and Multilingual Language Models** — https://aclanthology.org/2026.lrec-1.251/
- Lushtaku et al. (2026), **JudgeArena: A Unified Framework for Reproducible LLM-Judge Evaluation** — https://arxiv.org/abs/2608.02620

These resources scale the ideas in this notebook from a teaching micro-eval to research-grade evaluation.
